In [3]:
#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
import os
import requests

In [4]:
#------------------------------------------------ Begin_ fileName ----------------------------------------


regulatorName = 'ET NBE' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running ET NBE Web Scraping Tool v.1.1


In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

In [6]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

In [23]:
#------------------------------------------------ Begin_Main ----------------------------------------

regdict={
    #'ET NBE 1': 'https://nbe.gov.et/financial-institutions/banks/',
		#  'ET NBE 2': 'https://nbe.gov.et/financial-institutions/insurers/', 
		 'ET NBE 3': 'https://nbe.gov.et/financial-institutions/microfinance-institute/'
        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}



Typology ={

            'ET NBE 1': 'Banks',
            'ET NBE 2': 'Insurers',
            'ET NBE 3': "Microfinance Institutes",
}

headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36 Edg/141.0.0.0'
}


In [ ]:

for reg in regdict:

    print(f'Working with list {reg}')

    response = requests.get(regdict[reg], headers=headers)
    #print(response.status_code)

    

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')        
        pages = soup.find_all('div', class_='e-load-more-anchor')
        max_page = pages[0].get('data-max-page')

            
        if reg == 'ET NBE 3':
            for i in range(int(max_page)):
                page_url = 'https://nbe.gov.et/financial-institutions/microfinance-institute/'+str(i+1)
                response = requests.get(page_url, headers=headers)
                #print(response.status_code)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    data_cards = soup.find_all('div', attrs={'data-elementor-id': '33777'})

                    
                    for data_card in data_cards:
                        h1_ = data_card.find('h1')
                        company_name  = h1_.text
                        print(company_name)
                        sqldict['Name'].append(company_name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Cntry'].append('ET')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[-1])

                        lis = data_card.find_all('li')
                        for index,li in enumerate(lis):
                            #print(index,li.text.replace('\n',''))
                            if index == 0 :
                                ceo_name = li.text
                                ceo_name = ceo_name if ceo_name != 'Unknown' else ''
                                #print('CEO: ', ceo_name)
                            elif index ==1:
                                location_ = li.text
                                location_ = location_.strip(' ') if location_ != 'Unknown' else ''
                                sqldict['City'].append(location_.replace('\n',' ').replace('Unknown','').strip())
                                #print('Location: ',location_)
                            elif index == 2:
                                phone_ = li.text
                                phone_ = phone_ if phone_.strip(' ') != 'Unknown' else ''
                                print('Phone: ',phone_)
                                sqldict['Phone'].append(phone_.replace('\n',' ').replace('Unknown','').strip())
                            elif index == 3:
                                NBE_MFI_No_ = li.text
                                NBE_MFI_No_ = NBE_MFI_No_ if NBE_MFI_No_.strip(' ')!= 'Unknown' else ''
                                print('Internal: ', NBE_MFI_No_)
                                sqldict['InternalID_1_type'].append('NBE MFI NO')
                                sqldict['InternalID_1'].append(NBE_MFI_No_.split(':')[-1].replace('\n',' ').replace('Unknown','').strip())
                            elif index == 4:
                                fax_ = li.text
                                fax_ = fax_ if fax_.strip(' ') != 'Unknown' else ''
                                sqldict['Fax'].append(fax_.split('+')[-1].replace('Unknown','').strip())
                                print('Fax_: ',fax_)
                            elif index == 5:
                                tradeNr = li.text
                                tradeNr = tradeNr if tradeNr.strip(' ') != 'Unknown' else ''
                                sqldict['InternalID_2_type'].append('Trade No')
                                sqldict['InternalID_2'].append(tradeNr.split(':')[-1].replace('\n',' ').replace('Unknown','').strip())
                                print(tradeNr)
                            # elif index == 6:
                            #     establishment_date = li.text
                            #     #print(establishment_date)
                            # elif index == 7:
                            #     no_branch = li.text
                            #     #print(no_branch)
                            # elif index == 8:
                            #     pobox = li.text
                            #     #print(pobox)
                    
                        sqldict = bourange_same_length_array(sqldict)

        elif reg == 'ET NBE 2':
            for i in range(int(max_page)):
                page_url = 'https://nbe.gov.et/financial-institutions/insurers/'+str(i+1)
                response = requests.get(page_url, headers=headers)
                #print(response.status_code)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    data_cards = soup.find_all('div', attrs={'data-elementor-id': '33725'})
                    for data_card in data_cards:
                        h1_ = data_card.find('h1')
                        company_name  = h1_.text
                        print(company_name)
                        sqldict['Name'].append(company_name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Cntry'].append('ET')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[-1])
                        lis = data_card.find_all('li')
                        for index,li in enumerate(lis):
                            #print(index,li.text.replace('\n',''))

                            if index == 0:
                                phone_ = li.text
                                phone_ = phone_ if phone_.strip(' ') != 'Unknown' else ''
                                print('Phone: ',phone_)
                                sqldict['Phone'].append(phone_.replace('\n',' ').replace(' ',''))
                            elif index == 1:
                                email_ = li.text
                                email_ = email_ if email_.strip(' ') != 'Unknown' else ''
                                print('Email: ', email_)
                                sqldict['Email'].append(email_.replace('\n',' ').replace(' ',''))
                            elif index == 2:
                                fax_ = li.text
                                fax_ = fax_ if fax_.strip(' ') != 'Unknown' else ''
                                sqldict['Fax'].append(fax_.split('+')[-1].strip())
                                #print('Fax_: ',fax_)
                        sqldict = bourange_same_length_array(sqldict)
        
        elif reg == 'ET NBE 1':
            for i in range(int(max_page)):
                page_url = 'https://nbe.gov.et/financial-institutions/banks/'+str(i+1)
                response = requests.get(page_url, headers=headers)
                #print(response.status_code)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    data_cards = soup.find_all('div', attrs={'data-elementor-id': '24679'})
                    for data_card in data_cards:
                        h1_ = data_card.find('h1')
                        company_name  = h1_.text
                        #print(company_name)
                        sqldict['Name'].append(company_name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Cntry'].append('ET')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[-1])
                        lis = data_card.find_all('li')
                        for index,li in enumerate(lis):
                            #print(index,li.text.replace('\n',''))

                            if index == 0:
                                phone_ = li.text
                                #print(phone_)
                                phone_ = phone_ if phone_.strip(' ') != 'Unknown' else ''
                                phone_ = phone_.replace('\n',' ').strip()
                                sqldict['Phone'].append(phone_)
                            elif index == 1:
                                email_ = li.text
                                email_ = email_ if email_.strip(' ') != 'Unknown' else ''
                                email_ =  email_.replace('\n',' ').strip()
                                sqldict['Email'].append(email_)

                            elif index == 3:
                                swift_code = li.text
                                swift_code = swift_code if swift_code.strip(' ') != 'Unknown' else ''
                                #sqldict['BIC SWIFT Code'].append(swift_code)
                                swift_code =  swift_code.replace('\n',' ').strip('\n').lstrip(' ')
                                sqldict['BIC SWIFT Code'].append(swift_code)
                            elif index == 4:
                                website_ = li.text
                                website_ = website_ if website_.strip(' ') != 'Unknown' else ''
                                #sqldict['Website'].append(website_)                                
                                website_ = website_.replace('\n',' ').strip('\n').lstrip(' ')
                                sqldict['Website'].append(website_)
                            elif index == 7:
                                fax_ = li.text
                                fax_ = fax_ if 'Unknown' in fax_.strip(' ') else ''
                                fax_ = fax_.replace('\n',' ').strip('\n').lstrip(' ').split('+')[-1]
                                fax_ = fax_.replace('Unknown','')
                                sqldict['Fax'].append(fax_)
                        sqldict = bourange_same_length_array(sqldict)

    






Working with list ET NBE 3
Midre-Geez Microfinance Institution S.C
Phone:  

 
Unknown
Internal:  

 
NBE MFI No: MFI/072/2025
Fax_:  

 
Unknown


 
Unknown
Mefthe Microfinance S.C
Phone:  

 
0949077777
Internal:  

 
NBE MFI No: 52
Fax_:  

 
Unknown


 
Unknown
kefita Microfinance S.C
Phone:  

 
0911381242 , 0933695403
Internal:  

 
NBE MFI No: 51
Fax_:  

 
Unknown


 
Unknown
Rama Microfinance S.C
Phone:  

 
0911951484
Internal:  

 
NBE MFI No: 50
Fax_:  

 
Unknown


 
Unknown
Bilale Microfinance S.C
Phone:  

 
0911353890
Internal:  

 
NBE MFI No: 49
Fax_:  

 
Unknown


 
Unknown
Torban Microfinance S.C
Phone:  

 
0913626999 , 0913730239
Internal:  

 
NBE MFI No: 48
Fax_:  

 
Unknown


 
Unknown
Semien microfinance S.C
Phone:  

 
0914107403
Internal:  

 
NBE MFI No: 47
Fax_:  

 
Unknown


 
Unknown
Marchiwa Microfinance S.C
Phone:  

 
0911763263
Internal:  

 
NBE MFI No: 46
Fax_:  

 
Unknown


 
Unknown
Awra Amba Microfinance S.C
Phone:  

 
0916823282
Internal: 

In [26]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)
writer.save()
writer.close()
#driver.quit()
sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_21920\941835672.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [27]:
df.to_csv('list3_ver1.csv')